## Building a Research Note Agent Loop from scratch

This follows the same style as `5_extra.ipynb`: simple Python functions, hand-written tool schemas, and one loop where the LLM decides which tool to call next.

# What are we building?

A tiny research agent that can inspect local text files, save useful notes, and then answer a question from those notes.

This is a first-principles version of a common agentic workflow:

1. Look at available sources
2. Read relevant sources
3. Save notes
4. Answer from the notes

No agent framework required.

In [ ]:
# Start with imports - rich makes terminal output easier to read

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
from pathlib import Path
import json

load_dotenv(override=True)

In [ ]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [ ]:
openai = OpenAI()

In [ ]:
# The research sources live in the twin folder used by earlier labs

source_dir = Path("twin")
allowed_sources = sorted(path.name for path in source_dir.glob("*.txt"))

allowed_sources

In [ ]:
def list_sources() -> str:
    result = "Available sources:\n"
    for index, filename in enumerate(allowed_sources):
        path = source_dir / filename
        result += f"{index + 1}. {filename} ({path.stat().st_size} bytes)\n"
    show(result)
    return result

In [ ]:
def read_source(filename: str) -> str:
    if filename not in allowed_sources:
        return f"Source '{filename}' is not available. Use list_sources first."

    path = source_dir / filename
    content = path.read_text(encoding="utf-8")
    result = f"# Source: {filename}\n\n{content}"
    show(result[:1200] + ("\n\n[dim]...truncated in display...[/dim]" if len(result) > 1200 else ""))
    return result

In [ ]:
# This is the agent's notebook - the place where it saves what it learns

research_notes = []

In [ ]:
def save_note(source: str, note: str) -> str:
    research_notes.append({"source": source, "note": note})
    return get_notes()

In [ ]:
def get_notes() -> str:
    if not research_notes:
        result = "No research notes saved yet."
    else:
        result = "Research notes:\n"
        for index, item in enumerate(research_notes):
            result += f"{index + 1}. [{item['source']}] {item['note']}\n"
    show(result)
    return result

In [ ]:
# Quick check that the deterministic tools work before we let the LLM use them

research_notes = []
list_sources()
save_note("manual_check", "The note-saving tool works.")

In [ ]:
list_sources_json = {
    "name": "list_sources",
    "description": "List the local text files that are available for research",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

read_source_json = {
    "name": "read_source",
    "description": "Read one local source file by filename",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The exact filename to read, as returned by list_sources"
            }
        },
        "required": ["filename"],
        "additionalProperties": False
    }
}

In [ ]:
save_note_json = {
    "name": "save_note",
    "description": "Save a concise research note from a source that was inspected",
    "parameters": {
        "type": "object",
        "properties": {
            "source": {
                "type": "string",
                "description": "The filename or source name that supports the note"
            },
            "note": {
                "type": "string",
                "description": "A concise note, ideally one sentence, that captures useful evidence"
            }
        },
        "required": ["source", "note"],
        "additionalProperties": False
    }
}

get_notes_json = {
    "name": "get_notes",
    "description": "Return all research notes saved so far",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

In [ ]:
tools = [
    {"type": "function", "function": list_sources_json},
    {"type": "function", "function": read_source_json},
    {"type": "function", "function": save_note_json},
    {"type": "function", "function": get_notes_json}
]

In [ ]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results

In [ ]:
def loop(messages, max_turns=12):
    turn = 0
    done = False
    while not done and turn < max_turns:
        turn += 1
        response = openai.chat.completions.create(
            model="gpt-5.5",
            messages=messages,
            tools=tools
        )

        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True

    if not done:
        show("[red]The agent stopped because it reached the maximum number of turns.[/red]")
        return None

    answer = response.choices[0].message.content
    show(answer)
    return answer

In [ ]:
system_message = """
You are a research note agent.
You answer questions by inspecting local source files and saving notes before you answer.

Follow this workflow:
1. Use list_sources to see what evidence is available.
2. Read the most relevant source files with read_source.
3. Save concise evidence notes with save_note.
4. Use get_notes before your final answer.
5. Answer only from the notes and sources you inspected.

If the sources do not contain enough information, say what is missing.
Provide your final answer in Rich console markup without code blocks.
Do not ask the user questions or clarification; complete the research loop yourself.
"""

user_message = """
What are the strongest career themes in this profile, and what evidence supports them?
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message}
]

In [ ]:
# Run the research agent

research_notes = []
loop(messages)

# Try another research question

Change the `user_message` below and run the final cell again. This verifies that the same loop works for a different goal.

In [ ]:
user_message = """
What would be three strong talking points for an introductory client meeting?
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message}
]

research_notes = []
loop(messages)